# 🎓 AI WEEK 03 - DATA CLEANING & PREPROCESSING

**Learning Objectives:** 📚
- Import and analyze datasets
- Handle data quality issues
- Prepare data for machine learning models

---

## 📦 Step 1: Import Required Libraries

In [ ]:
# Standard data science libraries
import numpy as np  # 🔢 Numerical operations
import pandas as pd  # 📊 Data manipulation
import os

# Display all files in the input directory
print("📁 Available datasets:")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(f"  └─ {os.path.join(dirname, filename)}")

## 📂 Step 2: Load the Dataset

In [ ]:
file_path = "/kaggle/input/student-data-errors/student_data_with_issues.csv"
df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")

## 🔍 Step 3: Initial Data Exploration

In [ ]:
print("First 5 rows of the dataset:")
print(df.head())
print("\n" + "="*80 + "\n")

print("Dataset Information:")
df.info()

### 🚨 Issues Identified:

1. **Data Type Mismatch**: `Study_Hours_Per_Day` is `object` instead of `float64`
   - Contains text values like "one", "two", "three point five", etc.
2. **Missing Values**: Some columns don't have all 2007 entries
3. **Potential Duplicates**: Need to check for duplicate rows
4. **Invalid Values**: Some hour values might exceed 24 hours/day

---

## 🔧 Step 4: Fix Data Type Issues

### 📝 Convert Text Numbers to Numeric

In [ ]:
# Create mapping dictionary for text to numeric conversion
num_words = {
    1: "one", 2: "two", 3: "three", 4: "four", 5: "five",
    6: "six", 7: "seven", 8: "eight", 9: "nine", 10: "ten",
    11: "eleven", 12: "twelve", 13: "thirteen", 14: "fourteen",
    15: "fifteen", 16: "sixteen", 17: "seventeen", 18: "eighteen",
    19: "nineteen", 20: "twenty", 21: "twenty one",
    22: "twenty two", 23: "twenty three", 24: "twenty four"
}

# Build reverse mapping with decimal support
text_to_num = {}
for num, word in num_words.items():
    text_to_num[word] = num
    # Add decimal variations (e.g., "three point five" = 3.5)
    for d in range(1, 10):
        text_to_num[f"{word} point {num_words[d]}"] = num + d / 10

# Apply conversion
df['Study_Hours_Per_Day'] = df['Study_Hours_Per_Day'].apply(
    lambda x: text_to_num.get(str(x), x)
)
df['Study_Hours_Per_Day'] = pd.to_numeric(df['Study_Hours_Per_Day'], errors='coerce')

print("Study_Hours_Per_Day converted to numeric!")
print("\n Sample of converted data:")
print(df[['Student_ID', 'Study_Hours_Per_Day']].head(20))

## 🔍 Step 5: Check for Duplicates

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

if duplicate_count > 0:
    print(f"Original rows: {len(df)}")
    df = df.drop_duplicates()
    print(f"Rows after removing duplicates: {len(df)}")
    print("Duplicates removed!")
else:
    print("No duplicates found!")

## ⚠️ Step 6: Identify and Remove Impossible Values

### 🕐 Check for Hours > 24 per Day

In [ ]:
# Get all columns that represent hours per day
hours_cols = [col for col in df.columns if col.endswith("_Hours_Per_Day")]

print(f"Checking {len(hours_cols)} hour-based columns:")
for col in hours_cols:
    print(f"  • {col}")

# Ensure all are numeric
df[hours_cols] = df[hours_cols].apply(pd.to_numeric, errors='coerce')

# Find impossible values (> 24 hours)
bad_cells = (
    df[hours_cols]
    .where(df[hours_cols] > 24)
    .stack()
    .reset_index()
)
bad_cells.columns = ['row_index', 'column', 'value']

print(f"\n Found {len(bad_cells)} impossible values (>24 hours):")
if len(bad_cells) > 0:
    print(bad_cells)

# Remove rows with any value > 24
rows_before = len(df)
df = df[~(df[hours_cols] > 24).any(axis=1)]
rows_removed = rows_before - len(df)

print(f"\n Removed {rows_removed} rows with impossible values")
print(f" Remaining rows: {len(df)}")

## 🔍 Step 7: Handle Missing Values

In [ ]:
print("Missing values per column:")
missing_info = df.isnull().sum()
missing_info = missing_info[missing_info > 0].sort_values(ascending=False)

if len(missing_info) > 0:
    for col, count in missing_info.items():
        percentage = (count / len(df)) * 100
        print(f"  • {col}: {count} ({percentage:.2f}%)")
else:
    print("No missing values found!")

### Strategy 1️⃣: Drop Rows with Missing Critical Values

In [ ]:
# Drop rows where critical columns are missing
print(f"Rows before dropping missing values: {len(df)}")
df_cleaned = df.dropna().copy()
print(f"Rows after dropping missing values: {len(df_cleaned)}")
print(f"Rows removed: {len(df) - len(df_cleaned)}")

### Strategy 2️⃣: Use Interpolation for Missing Values (Alternative)

In [ ]:
# Alternative approach: fill missing values using interpolation
df_interpolated = df.copy()

for col in hours_cols:
    if df_interpolated[col].isnull().any():
        df_interpolated[col] = df_interpolated[col].interpolate(method='linear')
        
print("Missing values filled using linear interpolation!")
print(f"Rows preserved: {len(df_interpolated)}")

## 🏷️ Step 8: Encode Categorical Variables

### 📊 Ordinal Encoding for Stress Level

Stress levels have a natural order: **Low < Moderate < High**

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
import pandas as pd

# Ordinal encoding for Stress Level
ordinal_encoder = OrdinalEncoder(categories=[['Low', 'Moderate', 'High']])
df_cleaned['Stress_Level_Encoded'] = ordinal_encoder.fit_transform(df_cleaned[['Stress_Level']])

print("Stress Level Encoding:")
print(df_cleaned[['Stress_Level', 'Stress_Level_Encoded']].head())

### 🔄 One-Hot Encoding for Gender

Gender has no inherent order, so we use **one-hot encoding**

In [ ]:
# One-hot encode Gender
onehot_encoder = OneHotEncoder(sparse_output=False)
gender_encoded = onehot_encoder.fit_transform(df_cleaned[['Gender']])

# Get feature names and add to dataframe
gender_columns = onehot_encoder.get_feature_names_out(['Gender'])
gender_df = pd.DataFrame(gender_encoded, columns=gender_columns, index=df_cleaned.index)

# Concatenate with original dataframe and drop original Gender column
df_cleaned = pd.concat([df_cleaned, gender_df], axis=1)
df_cleaned = df_cleaned.drop('Gender', axis=1)

print("✅ Gender one-hot encoded!")
print(f"\n📊 New columns: {[col for col in df_cleaned.columns if 'Gender' in col]}")
print("\n👀 Sample data:")
print(df_cleaned.head())

## 📋 Step 9: Final Dataset Summary

In [ ]:
print("=" * 80)
print("🎉 DATA CLEANING COMPLETE!")
print("=" * 80)
print(f"\n📊 Final Dataset Shape: {df_cleaned.shape[0]} rows × {df_cleaned.shape[1]} columns")
print(f"\n✅ Data types:")
df_cleaned.info()

print("\n📈 Statistical Summary:")
print(df_cleaned.describe())

## 💾 Step 10: Save Cleaned Dataset

In [ ]:
# Save the cleaned dataset
output_path = '/kaggle/working/student_data_cleaned.csv'
df_cleaned.to_csv(output_path, index=False)
print(f"💾 Cleaned dataset saved to: {output_path}")

---

## 🎯 Key Takeaways

**What we learned:**
1. ✅ How to identify and fix data type issues
2. ✅ Detecting and removing duplicates
3. ✅ Handling impossible/invalid values
4. ✅ Strategies for dealing with missing data
5. ✅ Encoding categorical variables (ordinal and one-hot)

**The dataset is now ready for machine learning! 🚀**

---